In [17]:
import random
from pokerkit import *
from dotenv import load_dotenv
import json
from anthropic import Anthropic, beta_tool

In [18]:
load_dotenv()

client = Anthropic()

In [23]:
system_prompt = """You are a poker agent playing 3-handed Texas Hold'em No-Limit. 
Blinds are 1/2. You are Player 1 (Hero).
Think about pot odds, position, and hand strength before deciding.
Use the take_action tool to submit your decision."""

@beta_tool
def take_action(action: str, reasoning: str, amount: float = 0) -> str:
    """Submit your poker action for this turn and give a maximum 3 sentence explanation for why.

    Args:
        action: One of fold, check, call, raise
        amount: Raise amount in chips. Required if action is raise.
        reasoning: Reasoning for why you took a poker action. Maximum 3 sentences. 
    """
    return json.dumps({"status": "accepted", "action": action, "amount": amount, 'reasoning': reasoning})

def run_turn(state_json):
    messages = [{
        "role": "user",
        "content": f"""It's your turn to act. Here is the current game state:
    
    ```json
    {state_json}
    ```
    
    Decide your action."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[take_action],
        system=system_prompt,
        messages=messages
    )

    agent_action = None

    for message in runner:
        tokens = f"Input: {message.usage.input_tokens}, Output: {message.usage.output_tokens}"

        for block in message.content:
            if block.type == "tool_use" and block.name == "take_action":
                agent_action = block.input

    return agent_action, tokens

def street_converter(index):
    if index == 0:
        return 'preflop'
    elif index == 1:
        return 'flop'
    elif index == 2:
        return 'turn'
    elif index == 3:
        return 'river'
    
def get_visible_operations(state):
    visible = []
    for op in state.operations:
        if isinstance(op, HoleDealing):
            continue
        elif isinstance(op, CardBurning):
            # Never visible
            continue
        else:
            visible.append(op)
    return visible


In [48]:
state = NoLimitTexasHoldem.create_state(
    # automations
    (
        Automation.ANTE_POSTING,
    Automation.BET_COLLECTION,
    Automation.BLIND_OR_STRADDLE_POSTING,
    Automation.CARD_BURNING,
    # Automation.HOLE_DEALING, # re-enable for full randomness, but right now we're biasing for the agent
    # Automation.BOARD_DEALING,
    Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
    Automation.HAND_KILLING,
    Automation.CHIPS_PUSHING,
    Automation.CHIPS_PULLING,
    ),
    True,  # False for big blind ante, True otherwise
    0,  # ante
    (1000, 2000),  # blinds or straddles
    2000,  # min bet
    (150000, 200000, 250000),  # starting stacks
    3,  # number of players
)
# EXAMPLE OF ONE HAND

agent_cards = ['Ac', 'As']
agent_card_idx = 0
reasonings = []
# 2. The Training/Simulation Loop
while state.status:
    # DEALING PHASE: If the engine needs cards, give them
    if state.can_deal_hole():
        if state.hole_dealee_index == 0:
            state.deal_hole(agent_cards[agent_card_idx]) # bias the game towards the agent
            agent_card_idx += 1
        if state.hole_dealee_index != 0:
            state.deal_hole()
    elif state.can_deal_board():
        state.deal_board()
    
    # ACTION PHASE: If a player (Agent or Opponent) needs to move
    elif state.actor_index is not None:
        res = get_visible_operations(state)
        obs = {
            "your_index": 0,
            "pot": state.total_pot_amount,
            "board": state.board_cards,
            "hole": state.hole_cards[state.actor_index],
            "stacks": state.stacks,
            "bets": state.bets,
            "street": street_converter(state.street_index),
            "can_fold?": state.can_fold(),
            "can_check_or_call?": state.can_check_or_call(),
            "can_raise?": state.can_complete_bet_or_raise_to(),
            "how_much_to_call": state.checking_or_calling_amount,          # how much a call costs
            "action_history": res
        }
        
        if state.actor_index == 0: # Agent
            action = run_turn(obs)
            if action[0]['action'] in ['check', 'call']:
                state.check_or_call() 
            elif action[0]['action'] == 'raise':
                state.complete_bet_or_raise_to(action[0]['amount'])
            elif action[0]['action'] == 'fold':
                state.fold()
            reasonings.append([action[0]['action'], action[0]['reasoning']])
        else: 
            state.check_or_call()
            
    else:
        break



In [49]:
reasonings

[['raise',
  'I have pocket aces, the strongest starting hand. With excellent pot odds (1000 to call into a 5000 pot) and two opponents already in the pot, raising builds value and puts pressure on weaker holdings. This is a premium hand that warrants aggression to maximize profit.'],
 ['raise',
  'With pocket aces on a relatively dry flop, I should bet for value and control. A 6,000 chip bet (half pot) is standard and will likely get calls from weaker hands while maintaining flexibility for future streets.'],
 ['raise',
  "I hold pocket aces (Ac As) which is the strongest hand possible, and have shown strength with my prior bets. Both opponents have been passive, only calling. Betting again on the turn continues building the pot while I'm ahead, and the 12,000 bet (40% of pot) is a reasonable value bet to keep them engaged through the river while extracting chips with the best hand."],
 ['raise',
  'I hold pocket aces, the absolute best hand, on a rainbow board with no possible straig

In [50]:
state.operations

[BlindOrStraddlePosting(commentary=None, player_index=0, amount=1000),
 BlindOrStraddlePosting(commentary=None, player_index=1, amount=2000),
 HoleDealing(commentary=None, player_index=0, cards=(Ac,), statuses=(False,)),
 HoleDealing(commentary=None, player_index=1, cards=(Ks,), statuses=(False,)),
 HoleDealing(commentary=None, player_index=2, cards=(Js,), statuses=(False,)),
 HoleDealing(commentary=None, player_index=0, cards=(As,), statuses=(False,)),
 HoleDealing(commentary=None, player_index=1, cards=(Qd,), statuses=(False,)),
 HoleDealing(commentary=None, player_index=2, cards=(3s,), statuses=(False,)),
 CheckingOrCalling(commentary=None, player_index=2, amount=2000),
 CompletionBettingOrRaisingTo(commentary=None, player_index=0, amount=4000),
 CheckingOrCalling(commentary=None, player_index=1, amount=2000),
 CheckingOrCalling(commentary=None, player_index=2, amount=2000),
 BetCollection(commentary=None, bets=(4000, 4000, 4000)),
 CardBurning(commentary=None, card=7s),
 BoardDeali

In [ ]:
#EXAMPLE OF A FULL GAME

# Initial Setup
player_stacks = [150000, 100000, 200000] # Starting stacks for 3 players [cite: 85]
hand_count = 0

# The Global Game Loop: Runs until only one player has chips
while len([s for s in player_stacks if s > 0]) > 1:
    active_indices = [i for i, stack in enumerate(player_stacks) if stack > 0]

    # If only one player is left, the game is over
    if len(active_indices) < 2:
        break

# Filter stacks for the engine
    current_stacks = tuple(player_stacks[i] for i in active_indices)
    hand_count += 1
    print(f"\n--- STARTING HAND {hand_count} ---")
    
    # Initialize the specific hand state with current stacks
    state = NoLimitTexasHoldem.create_state(
        (
            Automation.ANTE_POSTING,
            Automation.BET_COLLECTION,
            Automation.BLIND_OR_STRADDLE_POSTING,
            Automation.CARD_BURNING,
            Automation.HOLE_DEALING,
            Automation.BOARD_DEALING,
            Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
            Automation.HAND_KILLING,
            Automation.CHIPS_PUSHING,
            Automation.CHIPS_PULLING,
        ),
        True, 0, (1000, 2000), 2000, tuple(current_stacks), len(active_indices)
    )
    #big blind ante, ante, blinds, min bet, stacks, players

    # Internal Decision Loop (Per-Turn) [cite: 19, 49]
    while state.status:
        if state.can_deal_hole():
            state.deal_hole()
        elif state.can_deal_board():
            state.deal_board()
        elif state.actor_index is not None:
            # Generate the observation for your LLM system [cite: 9, 85]
            obs = {
                "pot": state.total_pot_amount,
                "board": state.board_cards,
                "hole": state.hole_cards[state.actor_index],
                "history": state.operations 
            }
            
            # Action: Here you would plug in your LLM system or Equity Tool [cite: 22, 36]
            action_rand = random.random()
            if action_rand < 0.1 and state.can_complete_bet_or_raise_to():
    # 10% chance to raise to 3x the big blind
                min_raise = state.min_completion_betting_or_raising_to_amount
                max_raise = state.max_completion_betting_or_raising_to_amount # All-in
        
        # Raise to 2x the minimum required to keep it moving
                target_raise = min(min_raise * 2, max_raise)
                state.complete_bet_or_raise_to(target_raise)
            elif action_rand < 0.2 and state.can_fold():
    # 10% chance to fold
                state.fold()
            else:
                state.check_or_call()
        else:
            break

    # Update player stacks for the next hand based on payoffs [cite: 91, 92]
    # payoffs return the net change (e.g., [-2000, 4000, -2000])
    for i, global_idx in enumerate(active_indices):
        player_stacks[global_idx] += int(state.payoffs[i])
    
    print(f"Hand {hand_count} Results: {state.payoffs}")
    print(f"New Stacks: {player_stacks}")

print(f"\nGAME OVER after {hand_count} hands!")
print(f"Final Winner Stacks: {player_stacks}")


--- STARTING HAND 1 ---
Hand 1 Results: [12000, -6000, -6000]
New Stacks: [162000, 94000, 194000]

--- STARTING HAND 2 ---
Hand 2 Results: [-2000, 4000, -2000]
New Stacks: [160000, 98000, 192000]

--- STARTING HAND 3 ---
Hand 3 Results: [-2000, 8000, -6000]
New Stacks: [158000, 106000, 186000]

--- STARTING HAND 4 ---
Hand 4 Results: [20000, -10000, -10000]
New Stacks: [178000, 96000, 176000]

--- STARTING HAND 5 ---
Hand 5 Results: [-8000, -2000, 10000]
New Stacks: [170000, 94000, 186000]

--- STARTING HAND 6 ---
Hand 6 Results: [12000, -6000, -6000]
New Stacks: [182000, 88000, 180000]

--- STARTING HAND 7 ---
Hand 7 Results: [-6000, -6000, 12000]
New Stacks: [176000, 82000, 192000]

--- STARTING HAND 8 ---
Hand 8 Results: [-2000, -2000, 4000]
New Stacks: [174000, 80000, 196000]

--- STARTING HAND 9 ---
Hand 9 Results: [-2000, 4000, -2000]
New Stacks: [172000, 84000, 194000]

--- STARTING HAND 10 ---
Hand 10 Results: [4000, -2000, -2000]
New Stacks: [176000, 82000, 192000]

--- START